# Silver layer: automated normal-form (1NF / 2NF / 3NF) checks (optional)

> **Status: provided, not yet executed for the submitted run.** Silver follows the normalized TPC-H schema by design;
> key uniqueness and all foreign keys were verified by `03_validation` (silver phase). This notebook turns the
> normal-form argument into data-driven functional-dependency (FD) tests. Run it to produce `bda_silver.normalization_checks`.

**Method.** An FD `A -> B` holds iff no value of `A` maps to more than one value of `B`
(`MAX(COUNT(DISTINCT B)) = 1` per `A` group).

| Form | Test | Pass condition |
|---|---|---|
| 1NF | every column has an atomic type; the declared key is non-null and unique | no array/map/struct columns, 0 key violations |
| 2NF | for composite keys (`partsupp`, `lineitem`): does a *part* of the key determine a non-key column? | no partial FD holds |
| 3NF | does a non-key column (or a known composite of non-key columns) determine another non-key column? | no transitive FD holds, except documented derived columns |

**Documented exception.** `bda_silver.lineitem.revenue` and `margin` are *derived convenience columns*: recomputable
from `l_extendedprice`, `l_discount`, `l_quantity` and `partsupp.ps_supplycost`, validated against the source by the
revenue reconciliation in `03_validation`. They are expected to show up as transitive FDs here.

In [ ]:
# Portability: target catalog and source data location are parameters.
# Defaults reproduce the original setup (Databricks Free Edition); override them via widgets or job parameters.
import re
dbutils.widgets.text("catalog", "workspace", "Target catalog")
dbutils.widgets.text("source_schema", "samples.tpch", "Source data (catalog.schema)")
CATALOG = dbutils.widgets.get("catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError(f"invalid catalog: {CATALOG!r}")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*)?", SOURCE_SCHEMA):
    raise ValueError(f"invalid source_schema: {SOURCE_SCHEMA!r}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  source_schema={SOURCE_SCHEMA}")

In [ ]:
from datetime import datetime
from pyspark.sql import types as T

S = "bda_silver"
KEYS = {
    "region": ["r_regionkey"], "nation": ["n_nationkey"], "supplier": ["s_suppkey"],
    "customer": ["c_custkey"], "part": ["p_partkey"], "orders": ["o_orderkey"],
    "partsupp": ["ps_partkey", "ps_suppkey"], "lineitem": ["l_orderkey", "l_linenumber"],
}
EXCLUDE = {"ingestion_timestamp"}  # lineage metadata, not a business attribute
# composite determinants worth testing explicitly (A1, A2) -> B
COMPOSITE = {
    "lineitem": [(["l_extendedprice", "l_discount"], "revenue"), (["l_partkey", "l_quantity"], "l_extendedprice")],
}
DERIVED = {("lineitem", "revenue"), ("lineitem", "margin")}  # documented exception
results = []

def record(table, form, check, holds_bad, detail, exception=False):
    status = "EXCEPTION (documented)" if (holds_bad and exception) else ("FAIL" if holds_bad else "PASS")
    results.append((table, form, check, status, detail))
    print(f"[{status}] {table}: {form} {check} - {detail}")

def fd_holds(table, determinant, dependents):
    """Return {dependent: True if determinant -> dependent holds in the data}."""
    det = ", ".join(determinant)
    aggs = ", ".join(f"COUNT(DISTINCT {d}) AS n_{i}" for i, d in enumerate(dependents))
    maxes = ", ".join(f"MAX(n_{i})" for i in range(len(dependents)))
    row = spark.sql(f"SELECT {maxes} FROM (SELECT {det}, {aggs} FROM {S}.{table} GROUP BY {det})").first()
    return {d: (row[i] or 0) <= 1 for i, d in enumerate(dependents)}

## 1NF: atomic columns and a valid key

In [ ]:
for t, key in KEYS.items():
    schema = spark.table(f"{S}.{t}").schema
    nested = [f.name for f in schema if isinstance(f.dataType, (T.ArrayType, T.MapType, T.StructType))]
    record(t, "1NF", "atomic columns", bool(nested), f"non-atomic: {nested or 'none'}")
    k = ", ".join(key)
    bad = spark.sql(f"""SELECT (SELECT COUNT(*) FROM {S}.{t} WHERE {' OR '.join(c + ' IS NULL' for c in key)})
                             + (SELECT COUNT(*) FROM (SELECT {k} FROM {S}.{t} GROUP BY {k} HAVING COUNT(*) > 1))""").first()[0]
    record(t, "1NF", f"key ({k}) non-null and unique", bad > 0, f"{bad} violations")

## 2NF: no partial dependency on part of a composite key

In [ ]:
for t, key in KEYS.items():
    if len(key) < 2:
        record(t, "2NF", "single-column key", False, "2NF holds trivially")
        continue
    nonkey = [c for c in spark.table(f"{S}.{t}").columns if c not in key and c not in EXCLUDE]
    for part in key:
        held = [c for c, h in fd_holds(t, [part], nonkey).items() if h]
        record(t, "2NF", f"{part} -> non-key", bool(held), f"partial FDs: {held or 'none'}")

## 3NF: no transitive dependency between non-key columns

In [ ]:
for t, key in KEYS.items():
    nonkey = [c for c in spark.table(f"{S}.{t}").columns if c not in key and c not in EXCLUDE]
    for a in nonkey:
        others = [b for b in nonkey if b != a]
        if not others:
            continue
        held = [b for b, h in fd_holds(t, [a], others).items() if h]
        # a column that is itself unique (a candidate key) determines everything; that is not a 3NF violation
        unique = spark.sql(f"SELECT COUNT(DISTINCT {a}) = COUNT(*) FROM {S}.{t}").first()[0]
        bad = held and not unique
        exception = bool(held) and all((t, b) in DERIVED for b in held)
        record(t, "3NF", f"{a} -> other non-key", bad, f"transitive FDs: {held or 'none'}" + (" (candidate key)" if unique and held else ""), exception)
    for det, dep in COMPOSITE.get(t, []):
        held = fd_holds(t, det, [dep])[dep]
        record(t, "3NF", f"({', '.join(det)}) -> {dep}", held, "holds" if held else "does not hold", (t, dep) in DERIVED)

## Persist and summarize

In [ ]:
df = spark.createDataFrame(results, "table_name string, normal_form string, check_name string, status string, detail string")
from pyspark.sql.functions import current_timestamp

(df.withColumn("checked_at", current_timestamp())
   .write.mode("overwrite").saveAsTable(f"{S}.normalization_checks"))
display(df.groupBy("normal_form", "status").count().orderBy("normal_form", "status"))
display(df.filter("status != 'PASS'"))